## Task 1 — Setup

In [ ]:
import torch
import torchvision
from torchvision import models, transforms
from torchvision.models import (
    ResNet18_Weights,
    MobileNet_V3_Small_Weights,
    EfficientNet_B0_Weights,
)
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import time
import random
import copy
import os
from pathlib import Path
from sklearn.decomposition import PCA

os.makedirs("outputs", exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

DATA_DIR = Path("/kaggle/input/datasets/puneet6060/intel-image-classification/seg_test/seg_test")
LABELS = ["buildings", "forest", "glacier", "mountain", "sea", "street"]

random.seed(42)

In [ ]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    print(dirname)

In [ ]:
def load_images(n_per_class=10):
    """Load n images per class."""
    image_set = []

    for label in LABELS:
        class_dir = DATA_DIR / label
        paths = random.sample(list(class_dir.glob("*.jpg")), n_per_class)

        for path in paths:
            img = Image.open(path).convert("RGB")
            image_set.append((img, label))

    random.shuffle(image_set)
    return image_set


image_set = load_images(n_per_class=10)
print(f"Total images loaded: {len(image_set)}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.flatten()

for i, label in enumerate(LABELS):
    sample_img = next(img for img, true_label in image_set if true_label == label)
    axes[i].imshow(sample_img)
    axes[i].set_title(label)
    axes[i].axis("off")

plt.tight_layout()
plt.savefig("outputs/dataset_sample.png")
plt.show()

**Comment:**

A pretrained ImageNet model is a reasonable starting point for this dataset, but it is not a perfect final solution.

The dataset labels are broad scene categories, while ImageNet contains more specific object and scene labels. This means the model may still recognize useful visual patterns such as mountains, water, buildings, and trees, but its predictions may not directly match the six dataset classes.

Because of that, pretrained ImageNet models are useful for feature extraction and transfer learning, but they may need fine-tuning for this specific scene classification task.

The dataset labels are coarse and sometimes inconsistent, which introduces noise. Additionally, the pretrained ImageNet model predicts fine-grained classes that do not directly align with the dataset labels.

## Task 2 — Baseline Inference with ResNet18

In this section, we load a pretrained ResNet18 model and prepare it for inference.

We will use ImageNet weights, which means the model was trained on 1000 classes.

In [ ]:
# Load pretrained ResNet18
resnet_weights = ResNet18_Weights.DEFAULT
resnet = models.resnet18(weights=resnet_weights).to(device).eval()

# Preprocessing pipeline (VERY important)
resnet_preproc = resnet_weights.transforms()

# ImageNet class labels (1000 classes)
imagenet_classes = resnet_weights.meta["categories"]

# Count parameters
print(f"ResNet18 parameters: {sum(p.numel() for p in resnet.parameters()):,}")

ResNet18 is successfully loaded with pretrained ImageNet weights.
The model contains ~11.7 million parameters and is ready for inference.

In [ ]:
import torch.nn.functional as F

def run_inference(model, preprocess, image, device, class_labels, top_k=5):
    """
    Run inference on a PIL image and return the top-k predictions.
    Returns a list of (class_name, probability) tuples.
    """
    # Preprocess image, add batch dimension, and move it to the same device as the model
    input_tensor = preprocess(image).unsqueeze(0).to(device)

    # Inference only: no gradients needed
    with torch.no_grad():
        output = model(input_tensor)

    # Convert logits to probabilities
    probs = F.softmax(output[0], dim=0)

    # Get top-k probabilities and class indices
    top_probs, top_indices = torch.topk(probs, top_k)

    return [
        (class_labels[idx], prob.item())
        for idx, prob in zip(top_indices, top_probs)
    ]

In [ ]:
sample_img, sample_label = image_set[0]

sample_preds = run_inference(
    resnet,
    resnet_preproc,
    sample_img,
    device,
    imagenet_classes
)

print(f"True dataset label: {sample_label}")
print("Top-5 ImageNet predictions:")
for class_name, prob in sample_preds:
    print(f"{class_name:30s} {prob:.4f}")

### Observation

The model does not predict the dataset label directly (e.g., "buildings"),
but instead outputs more specific ImageNet classes such as "bell cote", "monastery", and "church".

This is expected because the model was trained on ImageNet, which contains fine-grained object categories,
while our dataset uses broader scene-level labels.

However, the predictions are still semantically related to the true class,
which shows that the model captures useful visual features.

In [ ]:
resnet_results = []

for img, true_label in image_set:
    preds = run_inference(
        resnet,
        resnet_preproc,
        img,
        device,
        imagenet_classes
    )

    resnet_results.append({
        "true_label": true_label,
        "top1_class": preds[0][0],
        "top1_prob": preds[0][1],
        "top5_classes": [p[0] for p in preds],
        "top5_probs": [p[1] for p in preds],
    })

print(f"Processed {len(resnet_results)} images.")

In [ ]:
# Overall mean top-1 probability
top1_probs = [result["top1_prob"] for result in resnet_results]

overall_mean = np.mean(top1_probs)
print(f"Overall mean top-1 probability: {overall_mean:.4f}")

In [ ]:
# Mean top-1 probability per class

class_confidence = {}

for label in LABELS:
    probs = [
        result["top1_prob"]
        for result in resnet_results
        if result["true_label"] == label
    ]
    class_confidence[label] = np.mean(probs)

# Print results
for label, mean_prob in class_confidence.items():
    print(f"{label:10s}: {mean_prob:.4f}")

In [ ]:
# Prepare data for boxplot
data_by_class = []

for label in LABELS:
    probs = [
        result["top1_prob"]
        for result in resnet_results
        if result["true_label"] == label
    ]
    data_by_class.append(probs)

# Create boxplot
plt.figure(figsize=(8, 6))
plt.boxplot(data_by_class, tick_labels=LABELS)
plt.title("ResNet18 Top-1 Confidence by Class")
plt.ylabel("Top-1 Probability")

plt.tight_layout()
plt.savefig("outputs/resnet18_confidence_by_class.png")
plt.show()

**Comment:**

High confidence does not necessarily mean high accuracy. A model can be confidently wrong, especially when the predicted class space does not align with the dataset labels.

In a production system, confidence scores can be used to determine when predictions are reliable. For example, predictions with confidence above a certain threshold (e.g., 0.7) could be accepted automatically, while lower-confidence predictions could be flagged for human review.

This helps balance automation and reliability in real-world applications.

## Task 3: Multi-Model Comparison

In this section, we compare ResNet18, MobileNetV3-Small, and EfficientNet-B0 on the same image set.

In [ ]:
# MobileNetV3-Small — designed for mobile and edge deployment
mobile_weights = MobileNet_V3_Small_Weights.DEFAULT
mobilenet = models.mobilenet_v3_small(weights=mobile_weights).to(device).eval()
mobile_preproc = mobile_weights.transforms()

# EfficientNet-B0 — designed to maximize accuracy per unit of compute
effnet_weights = EfficientNet_B0_Weights.DEFAULT
efficientnet = models.efficientnet_b0(weights=effnet_weights).to(device).eval()
effnet_preproc = effnet_weights.transforms()

# Print parameter counts for all three models
for name, m in [
    ("ResNet18", resnet),
    ("MobileNetV3-Small", mobilenet),
    ("EfficientNet-B0", efficientnet),
]:
    params = sum(p.numel() for p in m.parameters())
    print(f"{name:22s}  {params:>12,} parameters")

**Comment:**

MobileNetV3-Small has the smallest parameter count, which means it is more lightweight and likely faster to run, especially on limited hardware such as phones or edge devices.

A smaller model usually has lower capacity, so it may miss some complex patterns compared with a larger model.

ResNet18 and EfficientNet-B0 have more parameters, which may give them more representational power, but they can also require more memory and compute. For a phone, MobileNet is likely the better starting point. For a cloud server, a larger model may be acceptable if it provides better prediction quality.

In [ ]:
def collect_results(model, preprocess, image_set, device, class_labels):
    """Run inference on all images and store top-1 and top-5 results."""
    results = []

    for img, true_label in image_set:
        preds = run_inference(model, preprocess, img, device, class_labels)

        results.append({
            "true_label": true_label,
            "top1_class": preds[0][0],
            "top1_prob": preds[0][1],
            "top5_classes": [p[0] for p in preds],
            "top5_probs": [p[1] for p in preds],
        })

    return results


mobilenet_results = collect_results(
    mobilenet,
    mobile_preproc,
    image_set,
    device,
    imagenet_classes
)

effnet_results = collect_results(
    efficientnet,
    effnet_preproc,
    image_set,
    device,
    imagenet_classes
)

print(f"MobileNet results: {len(mobilenet_results)} images")
print(f"EfficientNet results: {len(effnet_results)} images")

In [ ]:
# Select one image from each class
comparison_samples = []
for label in LABELS:
    sample = next((img, true_label) for img, true_label in image_set if true_label == label)
    comparison_samples.append(sample)

fig, axes = plt.subplots(6, 4, figsize=(18, 24))

for row, (img, true_label) in enumerate(comparison_samples):
    # Column 1: image
    axes[row, 0].imshow(img)
    axes[row, 0].set_title(f"True label: {true_label}")
    axes[row, 0].axis("off")

    model_outputs = [
        ("ResNet18", run_inference(resnet, resnet_preproc, img, device, imagenet_classes)[:3]),
        ("MobileNetV3-Small", run_inference(mobilenet, mobile_preproc, img, device, imagenet_classes)[:3]),
        ("EfficientNet-B0", run_inference(efficientnet, effnet_preproc, img, device, imagenet_classes)[:3]),
    ]

    for col, (model_name, preds) in enumerate(model_outputs, start=1):
        text = "\n".join([f"{cls}: {prob:.3f}" for cls, prob in preds])
        axes[row, col].text(0.05, 0.5, text, fontsize=11, va="center")
        axes[row, col].set_title(model_name)
        axes[row, col].axis("off")

plt.tight_layout()
plt.savefig("outputs/model_comparison_grid.png")
plt.show()

**Comment:**

The three models do not always agree on their top-1 predictions, and in many cases their outputs differ significantly.

This is expected because the models were trained on ImageNet, which contains fine-grained object-level labels, while this dataset uses broad scene-level categories.

For example, instead of predicting "forest", models often predict objects within the scene such as "stone wall", "boathouse", or "viaduct". Similarly, for "glacier", models may focus on prominent elements like birds (predicting "airliner") or snowy mountains ("alp").

This disagreement suggests that different models focus on different visual features. Because of this, combining predictions from multiple models (an ensemble) could improve robustness.

Among the three models, EfficientNet-B0 often produces predictions that are more semantically aligned with the scene (e.g., "valley", "alp"), even if they are not exact matches to the dataset labels.

Why do pretrained models sometimes give incorrect predictions on a new dataset?
Because of label mismatch and domain differences.

## Task 4: Speed vs. Accuracy Tradeoff

In this section, we benchmark inference latency for all three models and compare their suitability for different deployment scenarios.

In [ ]:
def benchmark_model(model, preprocess, image_set, device, n_warmup=5):
    """
    Benchmark single-image inference speed.
    Returns mean latency in milliseconds per image.
    """
    # Warm up the GPU — the first few calls are slower due to CUDA initialization
    for img, _ in image_set[:n_warmup]:
        tensor = preprocess(img).unsqueeze(0).to(device)
        with torch.no_grad():
            _ = model(tensor)

    # Timed run — synchronize before and after to get accurate GPU timing
    if device.type == "cuda":
        torch.cuda.synchronize()

    start = time.time()

    for img, _ in image_set:
        tensor = preprocess(img).unsqueeze(0).to(device)
        with torch.no_grad():
            _ = model(tensor)

    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed = time.time() - start

    return (elapsed / len(image_set)) * 1000



In [ ]:
resnet_ms = benchmark_model(resnet, resnet_preproc, image_set, device)
mobile_ms = benchmark_model(mobilenet, mobile_preproc, image_set, device)
effnet_ms = benchmark_model(efficientnet, effnet_preproc, image_set, device)

print(f"ResNet18:           {resnet_ms:.2f} ms/image")
print(f"MobileNetV3-Small:  {mobile_ms:.2f} ms/image")
print(f"EfficientNet-B0:    {effnet_ms:.2f} ms/image")

In [ ]:
models_names = ["ResNet18", "MobileNetV3-Small", "EfficientNet-B0"]
latencies = [resnet_ms, mobile_ms, effnet_ms]

plt.figure(figsize=(8, 5))
plt.bar(models_names, latencies)

plt.title("Inference Latency (ms per image)")
plt.ylabel("Milliseconds per image")
plt.xlabel("Model")

for i, v in enumerate(latencies):
    plt.text(i, v + 0.2, f"{v:.2f}", ha='center')

plt.tight_layout()
plt.savefig("outputs/inference_speed.png")
plt.show()

**Observation:**

ResNet18 was the fastest model in this benchmark, followed by MobileNetV3-Small and then EfficientNet-B0.

This result is slightly unexpected because MobileNet is designed to be lightweight, but actual latency depends not only on parameter count, but also on model architecture, hardware, preprocessing, and GPU execution efficiency.

EfficientNet-B0 was the slowest in this test, which suggests that it may provide better feature quality at the cost of higher inference latency.

In [ ]:
summary = [
    ("ResNet18", 11689512, resnet_ms),
    ("MobileNetV3-Small", 2542856, mobile_ms),
    ("EfficientNet-B0", 5288548, effnet_ms),
]

print(f"{'Model':22s} {'Parameters':>15s} {'ms / image':>12s}")
print("-" * 50)

for name, params, latency in summary:
    print(f"{name:22s} {params:15,} {latency:12.2f}")

**Comment: Real-time constraint**

If the system must process 50 images per second, the maximum allowable latency per image is:

1000 ms / 50 = 20 ms per image.

Based on the benchmark results:

- ResNet18 (~4.03 ms) meets the requirement comfortably
- MobileNetV3-Small (~7.02 ms) also meets the requirement
- EfficientNet-B0 (~10.24 ms) still meets the requirement

All three models are fast enough for this real-time constraint.

**Comment: Deployment scenarios**

(a) High-throughput cloud pipeline:
EfficientNet-B0 would be a good choice because it offers better feature quality and accuracy, and cloud environments can handle higher latency.

(b) On-device mobile app:
MobileNetV3-Small is the best choice because it is lightweight and designed specifically for mobile and edge devices.

(c) Safety-critical system:
EfficientNet-B0 would be preferred because accuracy and robustness are more important than speed, and it provides stronger feature representations.

## Task 5: Pretrained Features as a Window into Transfer Learning

In [ ]:
import copy

# Remove classification head
feature_extractor = copy.deepcopy(resnet)
feature_extractor.fc = torch.nn.Identity()
feature_extractor = feature_extractor.to(device).eval()

In [ ]:
def extract_features(model, preprocess, image, device):
    """Extract feature vector from image"""
    tensor = preprocess(image).unsqueeze(0).to(device)

    with torch.no_grad():
        features = model(tensor)

    return features.squeeze().cpu().numpy()

In [ ]:
feature_vectors = []
true_labels = []

for img, label in image_set:
    feat = extract_features(feature_extractor, resnet_preproc, img, device)
    feature_vectors.append(feat)
    true_labels.append(label)

feature_matrix = np.array(feature_vectors)

print(f"Feature matrix shape: {feature_matrix.shape}")

In [ ]:
pca = PCA(n_components=2)
features_2d = pca.fit_transform(feature_matrix)

fig, ax = plt.subplots(figsize=(8, 6))
colors = plt.cm.tab10(np.linspace(0, 1, len(LABELS)))

for i, label in enumerate(LABELS):
    mask = np.array([l == label for l in true_labels])

    ax.scatter(
        features_2d[mask, 0],
        features_2d[mask, 1],
        label=label,
        color=colors[i],
        s=60,
        alpha=0.75
    )

ax.legend()
ax.set_title("ResNet18 Feature Embeddings (PCA to 2D)")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")

plt.tight_layout()
plt.savefig("outputs/feature_embeddings.png")
plt.show()

**Comment: Feature clustering**

Images from the same class generally tend to cluster together in the 2D feature space.

This indicates that the pretrained ResNet18 has already learned useful visual representations, even without being trained on this specific dataset. The model is able to group images with similar visual characteristics, such as forests, water scenes, and urban structures.

However, there is still some overlap between classes (e.g., mountain and glacier), which suggests that the model does not perfectly separate all scene types. This is expected because it was trained on ImageNet categories rather than these specific labels.

---

**Comment: Transfer learning strategy**

If I were adapting ResNet18 to a new task with only 500 labeled examples (such as X-ray classification), I would start with feature extraction.

In this approach, I would freeze all pretrained layers and train only a new final classification layer. This helps avoid overfitting, since the dataset is small, while still leveraging the strong visual features already learned by the pretrained model.

Fine-tuning could be considered later, once the new classification head is trained and if more data becomes available.


What does this plot show?

features cluster → model learned general representations
not perfect → dataset mismatch
can be improved with fine-tuning

## Stretch Goal: Fine-Tuning the Classification Head

In this optional section, we adapt ResNet18 from ImageNet's 1000 classes to our 6 scene classes by replacing and training only the final classification layer.

In [ ]:
import torch.nn as nn
import torch.optim as optim
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset

In [ ]:
NUM_CLASSES = len(LABELS)
TRAIN_DIR = Path("/kaggle/input/datasets/puneet6060/intel-image-classification/seg_train/seg_train")

# Start from the pretrained ResNet18
ft_model = copy.deepcopy(resnet)

# Freeze all pretrained layers
for param in ft_model.parameters():
    param.requires_grad = False

# Replace the final classification layer with a new 6-class head
ft_model.fc = nn.Linear(ft_model.fc.in_features, NUM_CLASSES)

# Move model to GPU/CPU
ft_model = ft_model.to(device)

trainable = sum(p.numel() for p in ft_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in ft_model.parameters())

print(f"Trainable: {trainable:,} of {total:,} total parameters ({100 * trainable / total:.2f}%)")

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

full_train = ImageFolder(TRAIN_DIR, transform=train_transforms)
print(f"Classes (alphabetical): {full_train.classes}")

random.seed(42)
imgs_per_class = 50
balanced_indices = []

for class_idx in range(NUM_CLASSES):
    indices = [i for i, (_, lbl) in enumerate(full_train.samples) if lbl == class_idx]
    balanced_indices.extend(random.sample(indices, min(imgs_per_class, len(indices))))

train_subset = Subset(full_train, balanced_indices)
train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)

print(f"Training on {len(train_subset)} images across {NUM_CLASSES} classes")

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(ft_model.fc.parameters(), lr=1e-3)

for epoch in range(3):
    ft_model.train()
    running_loss = 0.0
    correct = 0
    total_seen = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = ft_model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total_seen += images.size(0)

    print(
        f"Epoch {epoch + 1}/3 — "
        f"loss: {running_loss / total_seen:.4f}, "
        f"train acc: {correct / total_seen:.3f}"
    )

In [ ]:
ft_model.eval()
idx_to_label = full_train.classes

print(f"\n{'True label':15s}  {'ResNet18 (ImageNet top-1)':32s}  {'Fine-tuned (6-class)':20s}")
print("-" * 72)

for test_label in ["forest", "sea", "buildings"]:
    img = next(img for img, label in image_set if label == test_label)

    # Original ResNet18 outputs an ImageNet class name
    original_top1 = run_inference(
        resnet,
        resnet_preproc,
        img,
        device,
        imagenet_classes
    )[0][0]

    # Fine-tuned model outputs one of our 6 scene classes
    tensor = resnet_preproc(img).unsqueeze(0).to(device)

    with torch.no_grad():
        ft_out = ft_model(tensor)

    ft_prediction = idx_to_label[ft_out.argmax(dim=1).item()]

    print(f"{test_label:15s}  {original_top1:32s}  {ft_prediction:20s}")

**Comment: Fine-tuning results**

Only 3,078 out of 11,179,590 parameters were updated during fine-tuning, which is about 0.03% of the model. This shows that most of the learned visual knowledge lives in the pretrained convolutional layers, while the new final layer only learns how to map those features to our 6 scene classes.

The fine-tuned model correctly predicted some scene categories, such as `sea`, but it still confused `buildings` with `street`. The model sometimes focuses on specific patterns (e.g., “worm fence”) instead of the overall scene (“forest”). This is reasonable because the model was trained on only 300 images for 3 epochs.

To improve results, the next step would be to train for more epochs, use more labeled data, evaluate on a larger test set, and possibly fine-tune some deeper layers instead of only training the final classification head.

In a real application, the fine-tuned output format is more useful because it predicts the target classes directly (`forest`, `sea`, `buildings`, etc.) instead of ImageNet labels like `worm fence` or `bell cote`. This shows the practical value of fine-tuning even a single layer: the model output becomes aligned with the real business task.

## Task 6: Summary and Recommendation

### Model Comparison

Based on the comparison grid and latency benchmark, ResNet18 is the best starting point for this dataset.

ResNet18 was the fastest model in my benchmark at about 4.03 ms/image. MobileNetV3-Small was slower at about 7.02 ms/image, and EfficientNet-B0 was the slowest at about 10.24 ms/image.

Prediction quality was mixed for all three models because ImageNet labels do not directly match the six scene categories. However, ResNet18 produced reasonable scene-related predictions in many cases and had the best speed result.

### Confidence Calibration

From the ResNet18 confidence boxplot, the model was most confident on `sea`, `mountain`, and `glacier`.

It was least confident on `forest`, `buildings`, and `street`.

This mostly matches my intuition. Scenes like sea, mountains, and glaciers have strong visual patterns, while forests, buildings, and streets can vary a lot and may contain many different objects.

### Production Recommendation

I would suggest starting with ResNet18 because it had the best latency in this experiment and produced reasonable semantic predictions.

The production pipeline should include the same preprocessing steps used by the pretrained weights: resize, center crop, convert to tensor, and ImageNet normalization.

Before shipping, I would flag the main limitation: the pretrained ImageNet model does not directly predict the six target scene classes, so its output labels may not align with the business task.

For a real application, I would fine-tune the model on labeled examples from the six target classes and use confidence thresholds to send uncertain predictions to human review.